# Train a ~124M GPT-style model on Kaggle (2x T4)

**Before running:** Settings -> Accelerator = **GPU T4 x2**. Internet doesn't need to be on.

**What changed vs. the 30M notebook**, and why each one matters at this size:
- **RoPE (rotary position embeddings)** instead of a learned `pos_emb` table. Learned position
  embeddings only ever see positions up to `BLOCK_SIZE` during training and generalize poorly
  past that; RoPE is a relative-position scheme baked into attention itself (same one Llama,
  Qwen, Mistral use) and extrapolates to unseen lengths far better - relevant once you're at
  `BLOCK_SIZE=1024` and want headroom for a later multi-turn fine-tune.
- **RMSNorm** instead of `LayerNorm`. Same normalizing role, no mean-subtraction/bias term -
  cheaper and it's what every modern small LM uses instead of LayerNorm.
- **Config bumped to GPT-2-small's exact shape** (12 layers, 12 heads, 768 width, 1024 context)
  - this is what actually gets you to ~124M params, not the layer/head counts alone.
- **Everything else is unchanged on purpose**: same Kaggle-quota-aware checkpoint/resume,
  same 2-GPU `DataParallel`, same batch-size prober, same LR schedule. None of that was broken.

**Before this will actually behave like GPT-2 at this size, you also need bigger/more varied
pretraining data** (see the note in the Config cell) - the architecture change alone doesn't
buy you GPT-2-level fluency without the token-count and vocab-size changes discussed earlier.

**Where your data lives:** your `meta.json`/`tokenizer.json`/`train.bin`/`val.bin` dataset
should be attached under Input on the right sidebar. The first config cell searches
`/kaggle/input` recursively for `meta.json`.

**Checkpointing:** every `CKPT_MINUTES` (default 15) the model, optimizer state, and step count
are saved to `/kaggle/working`. Re-running the notebook resumes automatically - this is what
makes multi-session training work within Kaggle's 12-hour session cap / weekly GPU quota.
Click **Save Version** after each session so the checkpoint in Output survives.

In [ ]:
!pip install -q -U torch --index-url https://download.pytorch.org/whl/cu121 2>/dev/null || true
import torch
print(torch.__version__, "| CUDA:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("GPU count:", torch.cuda.device_count())

In [ ]:
import os, sys, math, json, time, glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------- FIND YOUR DATA (Kaggle) -------------------------
hits = glob.glob("/kaggle/input/**/meta.json", recursive=True)
assert hits, (
    "couldn't find meta.json anywhere under /kaggle/input. "
    "Make sure your dataset is attached (right sidebar -> Add Input), then run:\n"
    "  !find /kaggle/input -name meta.json\n"
    "and set DATA_DIR manually to the folder it's in."
)
if len(hits) > 1:
    print("found meta.json in multiple places, using the first one:")
    for h in hits:
        print(" ", h)
DATA_DIR = os.path.dirname(hits[0])
print("using data from:", DATA_DIR)

meta = json.load(open(f"{DATA_DIR}/meta.json"))
print(json.dumps(meta, indent=2))
VOCAB_SIZE = meta["vocab_size"]

## Config
12 layers, 768 width, 12 heads, context 1024, tied embeddings, RoPE, RMSNorm -> **~124M params**
at a ~32-50k vocab (this is GPT-2-small's exact layer/head/width shape).

**Important - this config alone does not make it "as good as GPT-2".** GPT-2 was trained on
~8-10B tokens of diverse web text with a 50,257-token vocab. If your `meta.json` above still
shows `vocab_size: 16384` and only ~1B tokens from `wiki_data_prep.ipynb`, that's the bigger
gap to close, not the architecture:
- Re-run `wiki_data_prep.ipynb` with `VOCAB_SIZE = 32768` (or higher) and a bigger
  `TARGET_TOKENS` (2-5B if your Kaggle quota allows) before training this.
- The tokenizer and `train.bin`/`val.bin` this notebook loads must come from that re-run -
  vocab size baked into the tokenizer has to match `meta.json`'s `vocab_size`, which is where
  `VOCAB_SIZE` below comes from automatically.

In [ ]:
# ------------------------- CONFIG -------------------------
TEST_RUN = False

BLOCK_SIZE = 1024         # context length (up from 512) - RoPE handles this length fine
N_LAYER = 12
N_HEAD = 12
N_EMBD = 768
DROPOUT = 0.0             # fine as long as tokens-per-param stays healthy (see note above)

BATCH_SIZE = 8            # lower starting point than the 30M run - 124M + 1024 context needs
                          # more memory per sample; the batch-size probe cell will adjust this
GRAD_ACCUM = 16           # keeps effective batch size reasonable despite the smaller per-step batch

MAX_LR = 6e-4             # lower than the 30M run's 1e-3 - larger models want a smaller peak LR
MIN_LR = MAX_LR * 0.1
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0

CKPT_MINUTES = 15
EVAL_EVERY_STEPS = 200
EVAL_ITERS = 50
SAMPLE_EVERY_STEPS = 500

if TEST_RUN:
    MAX_STEPS = 300
    WARMUP_STEPS = 20
    RUN_NAME = "test_124m"
else:
    # ---- SET THIS from your real token budget and measured tokens/sec (see the speed-test cell) ----
    # tokens_per_step = BATCH_SIZE * GRAD_ACCUM * n_gpu * BLOCK_SIZE (printed below).
    # MAX_STEPS = target_tokens // tokens_per_step
    MAX_STEPS = 20000
    WARMUP_STEPS = 500
    RUN_NAME = "run1_124m"

SAVE_DIR = f"/kaggle/working/ckpt_{RUN_NAME}"
os.makedirs(SAVE_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
n_gpu = torch.cuda.device_count()
print(f"device={device}, n_gpu={n_gpu}, save_dir={SAVE_DIR}")
print(f"effective batch (tokens/step) = {BATCH_SIZE * GRAD_ACCUM * max(1,n_gpu) * BLOCK_SIZE:,}")

## Data loading
Memmap so `train.bin`/`val.bin` are never fully loaded into RAM. Unchanged from the 30M notebook.

In [ ]:
train_data = np.memmap(f"{DATA_DIR}/train.bin", dtype=np.uint16, mode="r")
val_data = np.memmap(f"{DATA_DIR}/val.bin", dtype=np.uint16, mode="r")
print(f"train tokens: {len(train_data):,} | val tokens: {len(val_data):,}")

def get_batch(split, batch_size):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - BLOCK_SIZE - 1, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+BLOCK_SIZE].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+BLOCK_SIZE].astype(np.int64)) for i in ix])
    if device == "cuda":
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    else:
        x, y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch("train", 4)
print("batch shapes:", xb.shape, yb.shape, "| max id:", int(xb.max()), "(must be <", VOCAB_SIZE, ")")

## Model
Same nanoGPT-style structure as before, with two swaps:
- `RMSNorm` replaces `LayerNorm` (no mean-centering or bias, just RMS-scale + learned gain).
- `apply_rope` replaces the learned `pos_emb` table - rotates each head's query/key vectors by
  an angle that depends on position, so relative position is encoded directly in the dot
  product `q @ k^T` used by attention, instead of being added to the token embedding up front.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, n_embd, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(n_embd))
        self.eps = eps
    def forward(self, x):
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return x * rms * self.weight


def build_rope_cache(block_size, head_dim, base=10000.0, device="cpu"):
    # standard RoPE: pairs of dims (2i, 2i+1) rotated by angle theta_i * position
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2, device=device).float() / head_dim))
    t = torch.arange(block_size, device=device).float()
    freqs = torch.outer(t, inv_freq)                    # [block_size, head_dim/2]
    cos = torch.cat([freqs.cos(), freqs.cos()], dim=-1) # [block_size, head_dim]
    sin = torch.cat([freqs.sin(), freqs.sin()], dim=-1)
    return cos, sin


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(q, k, cos, sin):
    # q, k: [B, n_head, T, head_dim] ; cos/sin: [T, head_dim] -> broadcast over B, n_head
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    q = q * cos + rotate_half(q) * sin
    k = k * cos + rotate_half(k) * sin
    return q, k


class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head, self.n_embd, self.dropout = n_head, n_embd, dropout
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.attn_drop = dropout
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, rope_cos, rope_sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(self.n_embd, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k = apply_rope(q, k, rope_cos[:T], rope_sin[:T])
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                            dropout_p=self.attn_drop if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_drop(self.proj(y))

class MLP(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd, bias=False), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd, bias=False), nn.Dropout(dropout),
        )
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = RMSNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = RMSNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)
    def forward(self, x, rope_cos, rope_sin):
        x = x + self.attn(self.ln1(x), rope_cos, rope_sin)
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer, n_head, n_embd, dropout):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = RMSNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight   # weight tying: saves ~vocab_size*n_embd params

        head_dim = n_embd // n_head
        cos, sin = build_rope_cache(block_size, head_dim)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size
        x = self.drop(self.tok_emb(idx))   # no pos_emb added here - RoPE happens inside attention
        rope_cos = self.rope_cos.to(x.device)
        rope_sin = self.rope_sin.to(x.device)
        for blk in self.blocks:
            x = blk(x, rope_cos, rope_sin)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=0.8, top_k=50):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

def count_params(model):
    return sum(p.numel() for p in model.parameters())

model = GPT(VOCAB_SIZE, BLOCK_SIZE, N_LAYER, N_HEAD, N_EMBD, DROPOUT).to(device)
n_params = count_params(model)
print(f"parameters: {n_params:,} ({n_params/1e6:.1f}M)")
if n_params < 100e6 or n_params > 160e6:
    print("NOTE: this isn't landing near ~124M - check VOCAB_SIZE (from meta.json) and the "
          "N_LAYER/N_HEAD/N_EMBD/BLOCK_SIZE config above.")

if n_gpu > 1:
    model = nn.DataParallel(model)   # simple multi-GPU for Kaggle's 2x T4; splits the batch across GPUs
raw_model = model.module if n_gpu > 1 else model

## Optimizer, LR schedule, checkpoint/resume
Unchanged from the 30M notebook - same cosine schedule with warmup, same atomic checkpoint save, same auto-resume.

In [ ]:
decay, no_decay = [], []
for n, p in raw_model.named_parameters():
    (no_decay if p.dim() < 2 else decay).append(p)
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": WEIGHT_DECAY}, {"params": no_decay, "weight_decay": 0.0}],
    lr=MAX_LR, betas=(0.9, 0.95),
)

def get_lr(step):
    if step < WARMUP_STEPS:
        return MAX_LR * (step + 1) / WARMUP_STEPS
    if step >= MAX_STEPS:
        return MIN_LR
    prog = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    coeff = 0.5 * (1 + math.cos(math.pi * prog))
    return MIN_LR + coeff * (MAX_LR - MIN_LR)

CKPT_PATH = f"{SAVE_DIR}/latest.pt"

def save_ckpt(step, best_val):
    tmp = CKPT_PATH + ".tmp"
    torch.save({
        "step": step, "best_val": best_val,
        "model": raw_model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "config": dict(vocab_size=VOCAB_SIZE, block_size=BLOCK_SIZE, n_layer=N_LAYER,
                        n_head=N_HEAD, n_embd=N_EMBD, dropout=DROPOUT),
    }, tmp)
    os.replace(tmp, CKPT_PATH)

start_step, best_val = 0, float("inf")
if os.path.exists(CKPT_PATH):
    ck = torch.load(CKPT_PATH, map_location=device)
    raw_model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optimizer"])
    start_step, best_val = ck["step"], ck["best_val"]
    print(f"RESUMED from step {start_step} (best val loss so far: {best_val:.4f})")
else:
    print("no checkpoint found, starting fresh")

## Find a safe batch size (run this once, before the real run)
Finds the largest `BATCH_SIZE` that fits in GPU memory. 124M + 1024-token context uses
noticeably more memory per sample than the 30M/512-context run, so don't skip this - the
starting `BATCH_SIZE=8` above is a conservative guess, not a measured value.

In [ ]:
def try_batch_size(bs):
    try:
        xb, yb = get_batch("train", bs)
        optimizer.zero_grad(set_to_none=True)
        _, loss = model(xb, yb)
        loss.mean().backward()
        optimizer.step()
        if device == "cuda":
            torch.cuda.synchronize()
        return True
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            return False
        raise
    finally:
        optimizer.zero_grad(set_to_none=True)
        if device == "cuda":
            torch.cuda.empty_cache()

if device == "cuda":
    candidates = [4, 8, 12, 16, 24, 32, 48, 64]
    safe = None
    for bs in candidates:
        ok = try_batch_size(bs)
        print(f"BATCH_SIZE={bs}: {'OK' if ok else 'OUT OF MEMORY'}")
        if not ok:
            break
        safe = bs
    assert safe is not None, "even the smallest batch size (4) OOM'd - reduce N_EMBD/N_LAYER/BLOCK_SIZE"
    recommended = max(4, int(safe * 0.8))
    print(f"\nlargest working batch size: {safe} | recommended BATCH_SIZE (with headroom): {recommended}")
    print("If this differs from the BATCH_SIZE set in the config cell above, update it there and re-run from that cell.")

    model = GPT(VOCAB_SIZE, BLOCK_SIZE, N_LAYER, N_HEAD, N_EMBD, DROPOUT).to(device)
    if n_gpu > 1:
        model = nn.DataParallel(model)
    raw_model = model.module if n_gpu > 1 else model
    decay, no_decay = [], []
    for n, p in raw_model.named_parameters():
        (no_decay if p.dim() < 2 else decay).append(p)
    optimizer = torch.optim.AdamW(
        [{"params": decay, "weight_decay": WEIGHT_DECAY}, {"params": no_decay, "weight_decay": 0.0}],
        lr=MAX_LR, betas=(0.9, 0.95),
    )
    print("model and optimizer reset to a fresh state (probing steps discarded)")
else:
    print("no GPU detected - skipping batch size probe (CPU runs are for testing only)")

## Quick speed test
Run once on real data to get tokens/sec, then size `MAX_STEPS` for your target token budget or time window.

In [ ]:
@torch.no_grad()
def estimate_val_loss():
    model.eval()
    losses = []
    for _ in range(EVAL_ITERS):
        xb, yb = get_batch("val", BATCH_SIZE)
        _, loss = model(xb, yb)
        losses.append(loss.mean().item())
    model.train()
    return sum(losses) / len(losses)

model.train()
t0 = time.time()
n_tok = 0
for _ in range(20):
    for _ in range(GRAD_ACCUM):
        xb, yb = get_batch("train", BATCH_SIZE)
        _, loss = model(xb, yb)
        (loss.mean() / GRAD_ACCUM).backward()
        n_tok += xb.numel()
    optimizer.step(); optimizer.zero_grad(set_to_none=True)
dt = time.time() - t0
tok_per_sec = n_tok / dt
print(f"~{tok_per_sec:,.0f} tokens/sec  ({dt/20*1000:.0f} ms/step)")
if not TEST_RUN:
    total_hours = MAX_STEPS * BATCH_SIZE*GRAD_ACCUM*max(1,n_gpu)*BLOCK_SIZE / tok_per_sec / 3600
    print(f"time for MAX_STEPS={MAX_STEPS}: {total_hours:.1f} hours")
    print(f"  = roughly {total_hours/11:.1f} Kaggle sessions at ~11 usable hrs/session")
    print("If that's too long/short, adjust MAX_STEPS above and re-run from the config cell "
          "(this restarts the run - only do this before real training starts).")

## Train
Main loop. Checkpoints every `CKPT_MINUTES`, evaluates val loss periodically, prints a text sample. Safe to stop and re-run the whole notebook at any time - it resumes from the last checkpoint.

In [ ]:
def decode_sample(ids):
    from tokenizers import Tokenizer
    tok = Tokenizer.from_file(f"{DATA_DIR}/tokenizer.json")
    return tok.decode(ids)

def generate_sample():
    ctx = torch.zeros((1, 1), dtype=torch.long, device=device)
    out = raw_model.generate(ctx, max_new_tokens=100, temperature=0.8, top_k=50)[0].tolist()
    print("--- sample ---")
    print(decode_sample(out))
    print("--------------")

last_ckpt_time = time.time()
t_start = time.time()
step = start_step
while step < MAX_STEPS:
    lr = get_lr(step)
    for g in optimizer.param_groups:
        g["lr"] = lr

    optimizer.zero_grad(set_to_none=True)
    try:
        for _ in range(GRAD_ACCUM):
            xb, yb = get_batch("train", BATCH_SIZE)
            _, loss = model(xb, yb)
            (loss.mean() / GRAD_ACCUM).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            save_ckpt(step, best_val)
            raise RuntimeError(
                f"OOM at step {step} with BATCH_SIZE={BATCH_SIZE}. Checkpoint was saved at step {step}. "
                f"Lower BATCH_SIZE in the config cell (or re-run the batch-size probe cell), "
                f"then re-run from the config cell - it will resume from step {step}."
            ) from e
        raise
    step += 1

    if step % 20 == 0:
        elapsed = time.time() - t_start
        print(f"step {step}/{MAX_STEPS} | loss {loss.mean().item():.4f} | lr {lr:.2e} | {elapsed/60:.1f} min elapsed")

    if step % EVAL_EVERY_STEPS == 0 or step == MAX_STEPS:
        vl = estimate_val_loss()
        print(f"  [eval] step {step} | val loss {vl:.4f}")
        best_val = min(best_val, vl)

    if step % SAMPLE_EVERY_STEPS == 0:
        generate_sample()

    if time.time() - last_ckpt_time >= CKPT_MINUTES * 60 or step == MAX_STEPS:
        save_ckpt(step, best_val)
        last_ckpt_time = time.time()
        print(f"  [checkpoint] saved at step {step}")

print("done. final val loss (best seen):", best_val)

## After training
- Kaggle sessions cap at ~9-12 hrs and the free weekly GPU quota is limited - just re-run the
  whole notebook in your next session, it resumes from `ckpt_run1_124m/latest.pt` automatically.
- Click **Save Version** periodically so the checkpoint files in `/kaggle/working` survive
  between sessions.
- When `best_val` stops improving, you're done. Download `ckpt_run1_124m/latest.pt` and
  `tokenizer.json` from the Output panel.
- **This checkpoint format is compatible with the same `finetune_slm.ipynb` (v3) from before**
  - it reads `config` out of the checkpoint dynamically, so `CKPT_PATH`/`TOKENIZER_PATH` are the
  only things to point at your new 124M files. No fine-tuning code changes needed, since the
  fine-tuning notebook's own model class still needs to match the architecture here (RoPE +
  RMSNorm) - that one small update is worth doing before you fine-tune this checkpoint, and I
  can write that update next if you're ready for it.